# 📊 02 - Stock Data Ingestion
## Download Nifty 500 Stocks and Nifty Index Data from Yahoo Finance

---

**Notebook:** 02_stock_ingestion.ipynb  
**Project:** Financial Market Trend Forecasting  
**Data Source:** Yahoo Finance (yfinance)  
**Data Window:** 2023-01-01 to 2026-01-01

---

## Objective

Download and construct the raw stock price dataset for all Nifty 500 stocks and Nifty index.

## Output

- File: `Market_Data/raw/raw_stock_data.csv`
- Format: Long format with columns: Date, Ticker, Open, High, Low, Close, Volume

## 1. Setup and Imports

In [1]:
# Install required packages if not available
# !pip install yfinance pandas numpy tqdm

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import os
import warnings
from tqdm import tqdm
import time

warnings.filterwarnings('ignore')

print(f"Pandas version: {pd.__version__}")
print(f"yfinance version: {yf.__version__}")

Pandas version: 2.3.3
yfinance version: 1.2.0


In [3]:
# Configuration
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"

# Output paths
RAW_DATA_DIR = "../Market_Data/raw"
OUTPUT_FILE = os.path.join(RAW_DATA_DIR, "raw_stock_data.csv")

# Create directory if not exists
os.makedirs(RAW_DATA_DIR, exist_ok=True)

print(f"Data Window: {START_DATE} to {END_DATE}")
print(f"Output Path: {OUTPUT_FILE}")

Data Window: 2023-01-01 to 2026-01-01
Output Path: ../Market_Data/raw\raw_stock_data.csv


## 2. Nifty 500 Ticker List

We'll use a curated list of Nifty 500 tickers. The `.NS` suffix is required for Yahoo Finance to identify NSE (National Stock Exchange) stocks.

In [5]:
# Nifty 500 stock tickers (Complete Official List - 500 stocks)
# These are the NSE symbols without the .NS suffix
# Composition: Nifty 50 + Nifty Next 50 + Nifty Midcap 100 + Nifty Smallcap 250 + Additional

NIFTY_500_TICKERS = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK", "HINDUNILVR", "SBIN", "BHARTIARTL", "KOTAKBANK", "ITC",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "HCLTECH", "SUNPHARMA", "TITAN", "BAJFINANCE", "ULTRACEMCO", "WIPRO",
    "NESTLEIND", "NTPC", "POWERGRID", "M&M", "TATAMOTORS", "JSWSTEEL", "TATASTEEL", "ADANIENT", "ADANIPORTS", "ONGC",
    "COALINDIA", "TECHM", "BAJAJFINSV", "GRASIM", "INDUSINDBK", "DIVISLAB", "DRREDDY", "CIPLA", "BRITANNIA", "EICHERMOT",
    "APOLLOHOSP", "SBILIFE", "HDFCLIFE", "BPCL", "HEROMOTOCO", "TATACONSUM", "UPL", "HINDALCO", "BAJAJ-AUTO", "LTIM",
    "ADANIGREEN", "AMBUJACEM", "BANKBARODA", "BERGEPAINT", "BOSCHLTD", "CHOLAFIN", "COLPAL", "DABUR", "DLF", "GAIL",
    "GODREJCP", "HAVELLS", "ICICIPRULI", "ICICIGI", "INDUSTOWER", "IOC", "JINDALSTEL", "LICI", "LUPIN", "MARICO",
    "MCDOWELL-N", "NAUKRI", "PIDILITIND", "PNB", "SBICARD", "SHREECEM", "SIEMENS", "SRF", "TATAPOWER", "TORNTPHARM",
    "TRENT", "VEDL", "ZOMATO", "ZYDUSLIFE", "ABB", "ATGL", "AWL", "DMART", "IIFL", "INDIGO",
    "IRFC", "JIOFIN", "LODHA", "MAXHEALTH", "PAYTM", "PGHH", "POLYCAB", "TIINDIA", "VBL", "YESBANK",
    "ABCAPITAL", "ACC", "ALKEM", "ASHOKLEY", "ASTRAL", "AUROPHARMA", "BANDHANBNK", "BEL", "BHARATFORG", "BHEL",
    "BIOCON", "CANBK", "CONCOR", "COROMANDEL", "CUMMINSIND", "DALBHARAT", "DEEPAKNTR", "ESCORTS", "FEDERALBNK", "FORTIS",
    "GLAND", "GMRINFRA", "GUJGASLTD", "HAL", "HINDPETRO", "IDFCFIRSTB", "IGL", "INDHOTEL", "IRCTC", "JUBLFOOD",
    "LTTS", "M&MFIN", "MFSL", "MOTHERSON", "MPHASIS", "MRF", "MUTHOOTFIN", "NAM-INDIA", "NATIONALUM", "OBEROIRLTY",
    "OFSS", "PAGEIND", "PERSISTENT", "PETRONET", "PFC", "PIIND", "RAMCOCEM", "RECLTD", "SAIL", "SCHAEFFLER",
    "SONACOMS", "STARHEALTH", "SUNTV", "SUPREMEIND", "SYNGENE", "TATACOMM", "TATAELXSI", "TATACHEM", "TORNTPOWER", "TVSMOTOR",
    "UCOBANK", "UNIONBANK", "UBL", "VOLTAS", "WHIRLPOOL", "ZEEL", "CROMPTON", "CANFINHOME", "COFORGE", "CYIENT",
    "EMAMILTD", "EXIDEIND", "FLUOROCHEM", "GLAXO", "GODREJPROP", "GRINDWELL", "HDFCAMC", "HONAUT", "IPCALAB", "IRB",
    "KAJARIACER", "KANSAINER", "KEI", "LAURUSLABS", "LICHSGFIN", "LTF", "MANAPPURAM", "METROPOLIS", "MRPL", "NATCOPHARM",
    "NAVINFLUOR", "NMDC", "NYKAA", "OIL", "PRESTIGE", "PVRINOX", "RAJESHEXPO", "RELAXO", "ROUTE", "SANOFI",
    "SHRIRAMFIN", "SJVN", "SKFINDIA", "SOBHA", "SUMICHEM", "SUNDARMFIN", "SUNDRMFAST", "SUVENPHAR", "SYMPHONY", "TATAINVEST",
    "THERMAX", "TIMKEN", "TRIDENT", "UFLEX", "UJJIVANSFB", "VGUARD", "VINATIORGA", "WELCORP", "WESTLIFE", "WOCKPHARMA",
    "AEGISCHEM", "AFFLE", "AJANTPHARM", "ALKYLAMINE", "AMARAJABAT", "ANGELONE", "APLLTD", "APTUS", "ATUL", "ASTRAZEN",
    "BALRAMCHIN", "BASF", "BATAINDIA", "BDL", "BEML", "BLUESTARCO", "BRIGADE", "CGPOWER", "CESC", "COCHINSHIP",
    "CRISIL", "CREDITACC", "CUB", "AARTIIND", "AAVAS", "AIAENG", "ALLCARGO", "AMBER", "APARINDS", "APLAPOLLO",
    "ARVINDFASN", "ASHOKA", "AUBANK", "AVANTIFEED", "BALAMINES", "BALKRISIND", "BAYERCROP", "BBTC", "BIRLACORPN", "BLUEDART",
    "BSOFT", "CAMPUS", "CAMS", "CAPLIPOINT", "CARBORUNIV", "CASTROLIND", "CCL", "CDSL", "CENTURYTEX", "CENTURYPLY",
    "CERA", "CHAMBLFERT", "CHEMCON", "CHENNPETRO", "CLEAN", "DCMSHRIRAM", "DEEPAKFERT", "DELHIVERY", "DELTACORP", "DEVYANI",
    "DHANUKA", "DODLA", "DOMS", "ECLERX", "EDELWEISS", "EIDPARRY", "ELECON", "ELGIEQUIP", "EMCURE", "ENDURANCE",
    "ENGINERSIN", "EPL", "EQUITAS", "ERIS", "ESABINDIA", "FACT", "FINCABLES", "FINOLEX", "FIVESTAR", "GAEL",
    "GALAXYSURF", "GARFIBRES", "GATEWAY", "GENUSPOWER", "GHCL", "GICRE", "GILLETTE", "GLENMARK", "GOCOLORS", "GODREJAGRO",
    "GODREJIND", "GPIL", "GRANULES", "GRAPHITE", "GREENPLY", "GRSE", "GSFC", "GSPL", "GUFICBIO", "GULFOILLUB",
    "HAPPSTMNDS", "HATSUN", "HBLPOWER", "HEMIPROP", "HERITGFOOD", "HIKAL", "HINDCOPPER", "HLEGLAS", "HOMEFIRST", "HPL",
    "HUDCO", "IBREALEST", "ICRA", "IDFC", "IEX", "IIFLWAM", "INDIACEM", "INDIGRID", "INFIBEAM", "INTELLECT",
    "IOB", "IOLCP", "IONEXCHANG", "ISGEC", "ITI", "J&KBANK", "JAMNAAUTO", "JBCHEPHARM", "JCHAC", "JINDALSAW",
    "JKCEMENT", "JKLAKSHMI", "JKTYRE", "JMFINANCIL", "JSL", "JSWENERGY", "JSWHL", "JUBLINGREA", "JUSTDIAL", "JYOTHYLAB",
    "KALYANKJIL", "KEC", "KFINTECH", "KIOCL", "KIRLOSENG", "KNRCON", "KPITTECH", "KRBL", "KSB", "LATENTVIEW",
    "LAXMIMACH", "LEMONTREE", "LINDEINDIA", "LUMAXTECH", "LUXIND", "MAHABANK", "MAHSCOOTER", "MAHSEAMLES", "MAITHANALL", "MANINFRA",
    "MANKIND", "MAPMYINDIA", "MASTEK", "MEDPLUS", "MIDHANI", "MMTC", "MOIL", "MONTECARLO", "MOREPENLAB", "MTARTECH",
    "MUTHOOTCAP", "NBCC", "NCC", "NESCO", "NETWORK18", "NEWGEN", "NFL", "NHPC", "NIACL", "NIITLTD",
    "NILKAMAL", "NLCINDIA", "NOCIL", "NRBBEARING", "OLECTRA", "ORIENTCEM", "ORIENTELEC", "PCBL", "PEL", "PENIND",
    "PFIZER", "PHOENIXLTD", "PNCINFRA", "PNBHOUSING", "POLYMED", "POONAWALLA", "POWERINDIA", "PRSMJOHNSN", "PSB", "PTC",
    "QUESS", "RADICO", "RAIN", "RALLIS", "RANEENGINE", "RATNAMANI", "RAYMOND", "RBL", "REDINGTON", "RITES",
    "ROSSARI", "RVNL", "SAFARI", "SAKSOFT", "SAREGAMA", "SASKEN", "SHAKTIPUMP", "SHALBY", "SHANKARA", "SHILPAMED",
    "SHOPERSTOP", "SHREDIGCEM", "SHYAMMETL", "SJS", "SOLARINDS", "SONATSOFTW", "SOUTHBANK", "SPARC", "SPANDANA", "STARCEMENT",
    "STLTECH", "SUBROS", "SUDARSCHEM", "SUNDARAM", "SUNFLAG", "SUNTECK", "SUPRAJIT", "SURYAROSNI", "SWANENERGY", "SWSOLAR",
    "SYRMA", "TANLA", "TATATECH", "TEAMLEASE", "TECHNOE", "TEGA", "TIMESGTY", "TINPLATE", "TNPL", "TRIVENI",
    "TTKPRESTIG", "TV18BRDCST", "TVTODAY", "UNICHEMLAB", "UTIAMC", "VAIBHAVGBL", "VIPIND", "VISAKAIND", "VOLTAMP", "VSTIND",
    "WABAG", "WEALTH", "WEBELSOLAR", "WELSPUNIND", "WHEELS", "XPRO", "ZENSARTECH", "3MINDIA", "ABSLAMC", "ADANIENSOL",
    "ADANIPOWER", "AETHER", "AKZOINDIA", "ANURAS", "ASAHIINDIA", "BANARISUG", "BLISSGVS", "CHOLAHLDNG", "CONCORDBIO", "DATAPATTNS",
    "DREAMFOLKS", "FLAIR"
]

# Add .NS suffix for Yahoo Finance
nse_tickers = [f"{ticker}.NS" for ticker in NIFTY_500_TICKERS]

# Add Nifty 50 Index
INDEX_TICKER = "^NSEI"

print(f"Total Nifty 500 stocks to download: {len(nse_tickers)}")
print(f"Index ticker: {INDEX_TICKER}")
print(f"\nSample tickers: {nse_tickers[:5]}")

Total Nifty 500 stocks to download: 502
Index ticker: ^NSEI

Sample tickers: ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS']


## 3. Download Stock Data Function

In [ ]:
def download_stock_data(ticker, start_date, end_date, max_retries=3):
    """
    Download OHLCV data for a single ticker from Yahoo Finance.
    
    Args:
        ticker: Stock ticker symbol (with .NS suffix for NSE stocks)
        start_date: Start date string (YYYY-MM-DD)
        end_date: End date string (YYYY-MM-DD)
        max_retries: Number of retry attempts
    
    Returns:
        DataFrame with OHLCV data or None if download fails
    """
    for attempt in range(max_retries):
        try:
            # Download data
            stock = yf.Ticker(ticker)
            df = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            if df.empty:
                return None
            
            # Reset index to get Date as column
            df = df.reset_index()
            
            # Keep only required columns
            df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
            
            # Add ticker column
            df['Ticker'] = ticker.replace('.NS', '')  # Remove .NS suffix for cleaner data
            
            # Reorder columns
            df = df[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]
            
            return df
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait before retry
            else:
                print(f"Failed to download {ticker}: {str(e)}")
                return None
    
    return None

## 4. Download All Stocks Data

In [ ]:
def download_all_stocks(tickers, start_date, end_date):
    """
    Download data for all tickers and combine into single DataFrame.
    
    Args:
        tickers: List of ticker symbols
        start_date: Start date string
        end_date: End date string
    
    Returns:
        Combined DataFrame with all stock data
    """
    all_data = []
    successful = []
    failed = []
    
    print(f"Downloading data for {len(tickers)} stocks...")
    print(f"Date range: {start_date} to {end_date}")
    print("-" * 50)
    
    for ticker in tqdm(tickers, desc="Downloading"):
        df = download_stock_data(ticker, start_date, end_date)
        
        if df is not None and not df.empty:
            all_data.append(df)
            successful.append(ticker)
        else:
            failed.append(ticker)
        
        # Small delay to avoid rate limiting
        time.sleep(0.1)
    
    print("-" * 50)
    print(f"Successfully downloaded: {len(successful)} stocks")
    print(f"Failed to download: {len(failed)} stocks")
    
    if failed:
        print(f"\nFailed tickers: {failed[:20]}{'...' if len(failed) > 20 else ''}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df, successful, failed
    else:
        return pd.DataFrame(), successful, failed

In [ ]:
# Download all Nifty 500 stocks
stock_df, successful_tickers, failed_tickers = download_all_stocks(
    nse_tickers, START_DATE, END_DATE
)

print(f"\nStock data shape: {stock_df.shape}")

## 5. Download Nifty Index Data

In [ ]:
# Download Nifty 50 Index data
print(f"Downloading Nifty Index ({INDEX_TICKER})...")

index_df = download_stock_data(INDEX_TICKER, START_DATE, END_DATE)

if index_df is not None:
    # Use 'NIFTY50' as ticker name
    index_df['Ticker'] = 'NIFTY50'
    print(f"Nifty Index data shape: {index_df.shape}")
    print(f"Date range: {index_df['Date'].min()} to {index_df['Date'].max()}")
else:
    print("Failed to download Nifty Index data")
    index_df = pd.DataFrame()

In [ ]:
# Combine stock and index data
if not index_df.empty:
    combined_df = pd.concat([stock_df, index_df], ignore_index=True)
else:
    combined_df = stock_df.copy()

print(f"Combined data shape: {combined_df.shape}")

## 6. Data Processing

In [ ]:
def process_stock_data(df):
    """
    Process and clean the stock data.
    
    Processing rules:
    - Convert Date to datetime
    - Sort by Date and Ticker
    - Remove duplicate rows
    - Remove rows where Close is NaN
    - Keep only trading days
    
    Args:
        df: Raw stock DataFrame
    
    Returns:
        Processed DataFrame
    """
    print("Processing stock data...")
    print(f"Initial shape: {df.shape}")
    
    # Make a copy
    df = df.copy()
    
    # 1. Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Remove timezone info if present
    if df['Date'].dt.tz is not None:
        df['Date'] = df['Date'].dt.tz_localize(None)
    
    # 2. Sort by Date and Ticker
    df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)
    print(f"After sorting: {df.shape}")
    
    # 3. Remove duplicate rows
    initial_rows = len(df)
    df = df.drop_duplicates(subset=['Date', 'Ticker'], keep='first')
    duplicates_removed = initial_rows - len(df)
    print(f"Duplicates removed: {duplicates_removed}")
    
    # 4. Remove rows where Close is NaN
    initial_rows = len(df)
    df = df.dropna(subset=['Close'])
    nan_removed = initial_rows - len(df)
    print(f"Rows with NaN Close removed: {nan_removed}")
    
    # 5. Ensure numeric types
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 6. Final sort
    df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)
    
    print(f"Final shape: {df.shape}")
    
    return df

In [ ]:
# Process the combined data
processed_df = process_stock_data(combined_df)

## 7. Validation Checks

In [ ]:
def validate_stock_data(df):
    """
    Perform validation checks on the processed stock data.
    
    Args:
        df: Processed stock DataFrame
    """
    print("=" * 60)
    print("VALIDATION REPORT")
    print("=" * 60)
    
    # 1. Dataset shape
    print(f"\n1. Dataset Shape: {df.shape}")
    print(f"   - Total rows: {len(df):,}")
    print(f"   - Total columns: {len(df.columns)}")
    
    # 2. Unique tickers
    unique_tickers = df['Ticker'].nunique()
    print(f"\n2. Unique Tickers: {unique_tickers}")
    print(f"   - Sample tickers: {df['Ticker'].unique()[:10].tolist()}")
    
    # 3. Date range
    print(f"\n3. Date Range:")
    print(f"   - Start Date: {df['Date'].min()}")
    print(f"   - End Date: {df['Date'].max()}")
    print(f"   - Total trading days: {df['Date'].nunique()}")
    
    # 4. Missing values
    print(f"\n4. Missing Values:")
    missing = df.isnull().sum()
    for col in df.columns:
        pct = (missing[col] / len(df)) * 100
        print(f"   - {col}: {missing[col]:,} ({pct:.2f}%)")
    
    # 5. Data types
    print(f"\n5. Data Types:")
    for col in df.columns:
        print(f"   - {col}: {df[col].dtype}")
    
    # 6. Value ranges
    print(f"\n6. Value Ranges:")
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        print(f"   - {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")
    
    # 7. Sample rows
    print(f"\n7. Sample Rows:")
    print(df.head(10).to_string())
    
    print("\n" + "=" * 60)

In [ ]:
# Run validation
validate_stock_data(processed_df)

## 8. Per-Ticker Statistics

In [ ]:
# Calculate per-ticker statistics
ticker_stats = processed_df.groupby('Ticker').agg({
    'Date': ['min', 'max', 'count'],
    'Close': ['mean', 'std', 'min', 'max'],
    'Volume': 'mean'
}).round(2)

ticker_stats.columns = ['Start_Date', 'End_Date', 'Trading_Days', 
                        'Avg_Close', 'Std_Close', 'Min_Close', 'Max_Close',
                        'Avg_Volume']

print(f"Per-Ticker Statistics (showing first 20):")
print(ticker_stats.head(20).to_string())

In [ ]:
# Check for tickers with low data coverage
low_coverage = ticker_stats[ticker_stats['Trading_Days'] < 100]
print(f"\nTickers with less than 100 trading days: {len(low_coverage)}")
if len(low_coverage) > 0:
    print(low_coverage.to_string())

## 9. Save to CSV

In [ ]:
# Save processed data to CSV
processed_df.to_csv(OUTPUT_FILE, index=False)

# Verify saved file
file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB

print(f"\nData saved to: {OUTPUT_FILE}")
print(f"File size: {file_size:.2f} MB")

In [ ]:
# Verify by reading back
verification_df = pd.read_csv(OUTPUT_FILE, parse_dates=['Date'])

print(f"\nVerification - Read back data:")
print(f"Shape: {verification_df.shape}")
print(f"Columns: {verification_df.columns.tolist()}")
print(f"Date range: {verification_df['Date'].min()} to {verification_df['Date'].max()}")

## 10. Summary Report

In [ ]:
print("\n" + "=" * 60)
print("STOCK DATA INGESTION - FINAL SUMMARY")
print("=" * 60)

print(f"\n📊 DATA COLLECTION SUMMARY:")
print(f"   - Total stocks attempted: {len(nse_tickers)}")
print(f"   - Successfully downloaded: {len(successful_tickers)}")
print(f"   - Failed to download: {len(failed_tickers)}")
print(f"   - Nifty Index included: {'Yes' if 'NIFTY50' in processed_df['Ticker'].values else 'No'}")

print(f"\n📁 DATASET STATISTICS:")
print(f"   - Total rows: {len(processed_df):,}")
print(f"   - Unique tickers: {processed_df['Ticker'].nunique()}")
print(f"   - Date range: {processed_df['Date'].min().date()} to {processed_df['Date'].max().date()}")
print(f"   - Trading days: {processed_df['Date'].nunique()}")

print(f"\n📋 COLUMN STRUCTURE:")
print(f"   - Columns: {processed_df.columns.tolist()}")

print(f"\n❓ MISSING VALUES:")
for col in processed_df.columns:
    missing = processed_df[col].isnull().sum()
    if missing > 0:
        print(f"   - {col}: {missing:,} missing")
    else:
        print(f"   - {col}: No missing values")

print(f"\n💾 OUTPUT:")
print(f"   - File: {OUTPUT_FILE}")
print(f"   - Size: {file_size:.2f} MB")

print("\n" + "=" * 60)
print("✅ Stock data ingestion completed successfully!")
print("=" * 60)

---

## Next Steps

1. **Notebook 03**: Calculate technical indicators using TA-Lib
2. **Notebook 04**: Download global market data
3. **Notebook 05**: Fetch news sentiment from Alpha Vantage

---

**Next Notebook:** `03_technical_indicators.ipynb`